# Notebook 00 — Context

**Continual Learning in Context: RML Extension for CL-BENCH**

This notebook initializes the RML extension by reproducing the core CL-BENCH logic in a small, runnable form:

- stateful vs. stateless reward trajectories
- per-instance gain
- cumulative gain
- simple stability / plasticity decomposition
- drift / stale-context flags
- downloadable CSV + PNG artifacts

The goal is not to reproduce the full benchmark yet. The goal is to verify that the extension runs, saves outputs, and establishes the analysis vocabulary for later notebooks.

> Lab reports identify reproducible structure before analytic explanation.

## 0. Setup

This cell makes the notebook work from either:

- `rml/notebooks/00_context.ipynb`
- the repository root
- a copied Colab-style environment after path adjustment

The expected folder structure is:

```text
rml/
├── notebooks/
├── src/
├── figures/
├── results/
└── data/
```

In [ ]:
from pathlib import Path
import sys

# Resolve project paths.
NOTEBOOK_DIR = Path.cwd()

# If running from rml/notebooks, RML_ROOT is parent.
if NOTEBOOK_DIR.name == "notebooks":
    RML_ROOT = NOTEBOOK_DIR.parent
else:
    # Fallback: assume current directory is rml/ or repo root containing rml/.
    RML_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "src").exists() else NOTEBOOK_DIR / "rml"

SRC_DIR = RML_ROOT / "src"
RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
DATA_DIR = RML_ROOT / "data"

for directory in [RESULTS_DIR, FIGURES_DIR, DATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if str(RML_ROOT) not in sys.path:
    sys.path.insert(0, str(RML_ROOT))

print("RML_ROOT:", RML_ROOT.resolve())
print("SRC_DIR:", SRC_DIR.resolve())
print("RESULTS_DIR:", RESULTS_DIR.resolve())
print("FIGURES_DIR:", FIGURES_DIR.resolve())

## 1. Imports

Notebook 00 uses the thin `src/` layer you already added:

- `gain.py`
- `stability.py`
- `drift.py`
- `utils.py`

If this import succeeds, the extension scaffold is working.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.gain import compute_gain, cumulative_gain, average_gain
from src.stability import compute_stability, compute_plasticity
from src.drift import detect_drift
from src.utils import ensure_directory

print("Imports OK.")

## 2. CL-BENCH Context

CL-BENCH evaluates whether AI systems improve through sequential experience.

The central comparison is:

```text
stateful system   = sees prior experience
stateless system  = reset for each instance
```

The paper defines gain as:

\[
g_t = r^{sf}_t - r^{sl}_t
\]

where:

- \(r^{sf}_t\) is stateful reward at instance \(t\)
- \(r^{sl}_t\) is stateless reward at the same instance
- positive gain means accumulated context helped
- negative gain means accumulated context hurt

RML adds an interpretability layer:

```text
reward → gain → context advantage → stability / plasticity → stale context
```

## 3. Toy Rollout

This toy rollout is not benchmark data.

It is a minimal demonstration of the CL-BENCH metric idea:

- early instances start near stateless baseline,
- stateful reward improves after experience accumulates,
- a drift point introduces stale context,
- the system then adapts again.

Later notebooks will replace this toy rollout with real CL-BENCH logs or exported results.

In [ ]:
# Toy rollout with 12 instances.
instances = np.arange(1, 13)

# Stateless performance: no sequential learning, varies only by instance difficulty.
stateless_rewards = np.array([
    0.20, 0.24, 0.18, 0.26,
    0.22, 0.25, 0.23, 0.21,
    0.27, 0.24, 0.26, 0.25
])

# Stateful performance: improves, then suffers a drift/stale-context hit, then adapts.
stateful_rewards = np.array([
    0.20, 0.28, 0.31, 0.38,
    0.42, 0.47, 0.18, 0.24,
    0.39, 0.45, 0.49, 0.53
])

# Variant boundaries are zero-indexed positions where a new variant begins.
# Here: instance 1, instance 7, instance 9.
variant_boundaries = [0, 6, 8]

gains = np.array(compute_gain(stateful_rewards, stateless_rewards))

df = pd.DataFrame({
    "instance": instances,
    "stateful_reward": stateful_rewards,
    "stateless_reward": stateless_rewards,
    "gain": gains,
    "variant_boundary": [i in variant_boundaries for i in range(len(instances))]
})

df

## 4. Summary Metrics

This cell computes the core Notebook 00 outputs:

- total gain
- average gain
- stability estimate
- plasticity estimate
- drift flags

In [ ]:
total_gain = cumulative_gain(gains)
mean_gain = average_gain(gains)
stability = compute_stability(gains, variant_boundaries)
plasticity = compute_plasticity(gains, variant_boundaries)
drift_flags = detect_drift(gains, threshold=-0.05)

summary = pd.DataFrame([{
    "total_gain": total_gain,
    "average_gain": mean_gain,
    "stability": stability,
    "plasticity": plasticity,
    "drift_flags_zero_indexed": drift_flags,
    "drift_flags_instance_number": [i + 1 for i in drift_flags],
}])

summary

## 5. Plot: Stateful vs. Stateless

This plot is the simplest visual form of the CL-BENCH comparison.

- solid line: stateful reward
- dashed line: stateless reward
- vertical lines: variant boundaries / context shifts

Notebook 00 saves the figure to:

```text
rml/figures/00_context_stateful_vs_stateless.png
```

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(df["instance"], df["stateful_reward"], marker="o", label="Stateful")
ax.plot(df["instance"], df["stateless_reward"], marker="o", linestyle="--", label="Stateless")

for boundary in variant_boundaries:
    ax.axvline(boundary + 1, linestyle=":", alpha=0.7)

ax.set_title("Notebook 00 — Stateful vs. Stateless Reward")
ax.set_xlabel("Instance")
ax.set_ylabel("Reward")
ax.set_xticks(instances)
ax.legend()
ax.grid(True, alpha=0.25)

figure_path = FIGURES_DIR / "00_context_stateful_vs_stateless.png"
fig.tight_layout()
fig.savefig(figure_path, dpi=160)

figure_path

## 6. Plot: Gain as Context Signal

Gain is interpreted here as a **context signal**.

```text
positive gain  → accumulated context helped
zero gain      → accumulated context did not matter
negative gain  → accumulated context hurt
```

Notebook 00 saves the figure to:

```text
rml/figures/00_context_gain_signal.png
```

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(df["instance"], df["gain"])
ax.axhline(0, linewidth=1)

for boundary in variant_boundaries:
    ax.axvline(boundary + 1, linestyle=":", alpha=0.7)

ax.set_title("Notebook 00 — Gain as Context Signal")
ax.set_xlabel("Instance")
ax.set_ylabel("Gain = stateful - stateless")
ax.set_xticks(instances)
ax.grid(True, axis="y", alpha=0.25)

gain_figure_path = FIGURES_DIR / "00_context_gain_signal.png"
fig.tight_layout()
fig.savefig(gain_figure_path, dpi=160)

gain_figure_path

## 7. Stale Context Flags

A negative gain event is not automatically drift, but it is a useful flag.

In later notebooks, this becomes an RML diagnostic:

```text
negative gain + variant shift + reused stale assumption → stale context
```

For Notebook 00, we simply mark large negative gain values.

In [ ]:
df["stale_context_flag"] = df.index.isin(drift_flags)

df[[
    "instance",
    "stateful_reward",
    "stateless_reward",
    "gain",
    "variant_boundary",
    "stale_context_flag",
]]

## 8. Save Results

Notebook 00 writes two CSV files:

```text
rml/results/00_context_rollout.csv
rml/results/00_context_summary.csv
```

In [ ]:
rollout_path = RESULTS_DIR / "00_context_rollout.csv"
summary_path = RESULTS_DIR / "00_context_summary.csv"

df.to_csv(rollout_path, index=False)
summary.to_csv(summary_path, index=False)

print("Saved:")
print("-", rollout_path)
print("-", summary_path)
print("-", figure_path)
print("-", gain_figure_path)

## 9. Downloadable Artifact Pack

This cell creates a zip file containing the Notebook 00 outputs:

```text
rml/results/00_context_artifacts.zip
```

In Jupyter, the path is printed.  
In Colab, the file can be downloaded directly if `google.colab` is available.

In [ ]:
import zipfile

zip_path = RESULTS_DIR / "00_context_artifacts.zip"

artifact_paths = [
    rollout_path,
    summary_path,
    figure_path,
    gain_figure_path,
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in artifact_paths:
        zf.write(path, arcname=path.name)

print("Artifact pack created:")
print(zip_path.resolve())

# Optional Colab download.
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Not running in Colab, or download unavailable. Use the printed path above.")

## 10. Lab Report Note

Notebook 00 establishes the reusable metric vocabulary for the RML extension:

```text
reward
gain
stateful vs. stateless
stability
plasticity
drift
stale context
```

The later notebooks expand each lane:

```text
01 → gain signal
07 → latent structure
11 → stateful vs. stateless
13 → stability
17 → plasticity
19 → stale context
23 → drift adaptation
29 → failure modes
```

**Continual Learning in Context** means that improvement is not merely memory.

Improvement requires reusable structure, valid context, and adaptation when that context changes.